# 07g — BCL POI 2008 normalization + lexical-context alignment

Uses the Stage-07f provenance/CRS-gated FileGDB without re-downloading it.

CP2-v2 no longer has a Beijing geofence, so this stage does **anchor-neighborhood extraction** for the temporally eligible 2007–2009 recurring non-HOME anchors rather than a hard-coded Beijing-only crop.

BCL provides `PNAME` + coordinates but no trusted category field. Raw names are preserved privately; only predeclared lexical signals are audited. No HOME/WORK/OFFICE/occupation labels are produced.


In [ ]:
from pathlib import Path
import importlib.util
import json
import os
import subprocess
import sys
import pandas as pd

REPO_URL='https://github.com/tanh1c/geolife.git'
BRANCH=os.environ.get('GEOLIFE_REPO_BRANCH','main')
REPO_DIR=Path(os.environ.get('GEOLIFE_REPO_DIR','/root/geolife'))

def run(cmd,cwd=None):
    subprocess.run(cmd,cwd=cwd,check=True)

if not (REPO_DIR/'.git').exists():
    run(['git','clone','--branch',BRANCH,'--single-branch',REPO_URL,str(REPO_DIR)])
else:
    run(['git','-C',str(REPO_DIR),'fetch','origin',BRANCH])
    run(['git','-C',str(REPO_DIR),'checkout',BRANCH])
    run(['git','-C',str(REPO_DIR),'reset','--hard',f'origin/{BRANCH}'])

run([sys.executable,'-m','pip','install','-q','-e','.[dev]'],cwd=REPO_DIR)
for p in (str(REPO_DIR),str(REPO_DIR/'src')):
    if p not in sys.path: sys.path.insert(0,p)

spec=importlib.util.spec_from_file_location('s07g',REPO_DIR/'analysis'/'07g_bcl_poi_2008_alignment.py')
assert spec is not None and spec.loader is not None
s07g=importlib.util.module_from_spec(spec)
sys.modules[spec.name]=s07g
spec.loader.exec_module(s07g)

print('sha:',subprocess.run(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip())
print(s07g.synthetic_self_check())


In [ ]:
V=Path('/mnt/geolife-data')
C07C=V/'cache'/'cp2_v2'/'07c_historical_source_alignment'
C07F=V/'cache'/'07f_bcl_poi_2008_acquisition'
C=V/'cache'/'cp2_v2'/'07g_bcl_poi_2008_alignment'; C.mkdir(parents=True,exist_ok=True)

ANCHORS=C07C/'anchor_observation_dates_private.pkl'
MANIFEST=C07F/'acquisition_manifest.json'
WORK=V/'cache'/'cp2_v2'/'05b_home_consensus_adaptive_work'/'adaptive_work_patterns_42_private.pkl'
PROFILE=V/'cache'/'cp2_v2'/'07b_factorized_work_profiles'/'factorized_work_profiles_private.pkl'
POI_PRIVATE=C/'bcl_neighborhood_pois_private.csv'

for required in [ANCHORS,MANIFEST,WORK,PROFILE]:
    if not required.exists(): raise FileNotFoundError(f'Run upstream stage first: {required}')

anchors=pd.read_pickle(ANCHORS)
work=pd.read_pickle(WORK)
profiles=pd.read_pickle(PROFILE)
manifest=json.loads(MANIFEST.read_text(encoding='utf-8'))
gate=s07g.validate_07f_manifest(manifest)
display(s07g.manifest_gate_frame(gate))
if not gate.ready: raise RuntimeError(f'Stage 07f manifest does not permit normalization: {gate.notes}')

eligible=s07g.bcl_eligible_anchors(anchors)
eligibility=s07g.summarize_bcl_eligibility(anchors,eligible)
display(eligibility)
assert len(eligible)==259, f'Expected frozen CP2-v2 BCL-relevant anchors=259, got {len(eligible)}'
assert eligible['user_id'].nunique()==23, f'Expected 23 BCL-relevant users, got {eligible.user_id.nunique()}'

tiles=s07g.build_spatial_tiles(eligible,padding_m=s07g.EXTRACTION_PADDING_M)
print('query tiles:',len(tiles),'eligible anchors:',len(eligible),'users:',eligible.user_id.nunique())
print('GDB:',gate.container_path)


In [ ]:
import modal

volume_name=os.environ.get('GEOLIFE_MODAL_VOLUME_NAME','geolife-data')
data_volume=modal.Volume.from_name(volume_name,create_if_missing=False)
app=modal.App('geolife-stage07g-bcl-poi-2008')
image=(modal.Image.debian_slim(python_version=f'{sys.version_info.major}.{sys.version_info.minor}')
       .apt_install('gdal-bin')
       .pip_install('pandas'))

@app.function(image=image,volumes={'/mnt/geolife-data':data_volume},timeout=1800,serialized=True)
def extract_bcl_neighborhoods(payload: dict) -> dict:
    from hashlib import sha256
    from pathlib import Path
    import subprocess
    import tempfile
    import pandas as pd

    def digest(path):
        h=sha256()
        with Path(path).open('rb') as f:
            for chunk in iter(lambda:f.read(8*1024*1024),b''): h.update(chunk)
        return h.hexdigest()

    source=Path(payload['source_path'])
    if not source.exists():
        return {'ok':False,'error':f'Stage-07f source file missing: {source}'}
    expected_sha=(payload.get('source_sha256') or '').lower()
    actual_sha=digest(source)
    if expected_sha and actual_sha.lower()!=expected_sha:
        return {'ok':False,'error':'Stage-07f source SHA-256 mismatch','actual_sha256':actual_sha}

    gdb=Path(payload['gdb_path'])
    if not gdb.exists() or not gdb.is_dir():
        candidates=sorted(Path('/mnt/geolife-data/external/bcl_poi_2008/extracted').rglob('*.gdb'))
        if len(candidates)!=1:
            return {'ok':False,'error':f'Expected one FileGDB, found {len(candidates)}','candidates':[str(x) for x in candidates[:20]]}
        gdb=candidates[0]

    layer=payload['layer']
    info=subprocess.run(['ogrinfo','-ro','-so',str(gdb),layer],capture_output=True,text=True)
    if info.returncode!=0:
        return {'ok':False,'error':'ogrinfo layer validation failed','stderr':info.stderr[:5000]}
    info_text=info.stdout
    for token in ('Geometry: Point','PNAME','X','Y'):
        if token not in info_text:
            return {'ok':False,'error':f'Expected layer token missing: {token}','ogrinfo':info_text[:12000]}

    frames=[]; raw_rows=0; failed=[]
    with tempfile.TemporaryDirectory() as tmp:
        tmp=Path(tmp)
        for idx,tile in enumerate(payload['tiles']):
            out=tmp/f'tile_{idx:04d}.csv'
            cmd=[
                'ogr2ogr','-f','CSV',str(out),str(gdb),layer,
                '-spat',str(tile['xmin']),str(tile['ymin']),str(tile['xmax']),str(tile['ymax']),
                '-select','PNAME,X,Y','-overwrite',
            ]
            cp=subprocess.run(cmd,capture_output=True,text=True)
            if cp.returncode!=0:
                failed.append({'tile_id':tile['tile_id'],'stderr':cp.stderr[:1000]})
                continue
            if not out.exists() or out.stat().st_size==0:
                continue
            try:
                frame=pd.read_csv(out)
            except pd.errors.EmptyDataError:
                continue
            if frame.empty: continue
            required={'PNAME','X','Y'}
            if not required.issubset(frame.columns):
                failed.append({'tile_id':tile['tile_id'],'stderr':f'CSV missing columns: {sorted(required-set(frame.columns))}'})
                continue
            raw_rows+=len(frame)
            frame=frame[['PNAME','X','Y']].rename(columns={'PNAME':'poi_name','X':'longitude','Y':'latitude'})
            frames.append(frame)

    if failed:
        return {'ok':False,'error':'one or more GDAL tile extracts failed','failed_tiles':failed[:20],'failed_tile_count':len(failed)}

    pois=(pd.concat(frames,ignore_index=True) if frames else pd.DataFrame(columns=['poi_name','longitude','latitude']))
    pois['poi_name']=pois['poi_name'].fillna('').astype(str)
    pois['longitude']=pd.to_numeric(pois['longitude'],errors='coerce')
    pois['latitude']=pd.to_numeric(pois['latitude'],errors='coerce')
    pois=pois.dropna(subset=['longitude','latitude']).drop_duplicates(['poi_name','longitude','latitude']).reset_index(drop=True)

    output=Path(payload['output_path']); output.parent.mkdir(parents=True,exist_ok=True)
    pois.to_csv(output,index=False)
    data_volume.commit()
    return {
        'ok':True,
        'source_sha256':actual_sha,
        'gdb_path':str(gdb),
        'layer':layer,
        'tiles':len(payload['tiles']),
        'raw_tile_rows':int(raw_rows),
        'deduplicated_pois':int(len(pois)),
        'output_path':str(output),
        'poi_records':pois.to_dict(orient='records'),
    }

verification=manifest.get('file_verification') or {}
payload={
    'source_path':manifest.get('local_path'),
    'source_sha256':verification.get('sha256'),
    'gdb_path':gate.container_path,
    'layer':s07g.EXPECTED_LAYER,
    'tiles':tiles.to_dict(orient='records'),
    'output_path':str(POI_PRIVATE),
}
with app.run():
    extraction=extract_bcl_neighborhoods.remote(payload)
if not extraction.get('ok'):
    raise RuntimeError(extraction)

poi_records=extraction.pop('poi_records',[])
display(pd.DataFrame([extraction]))
pois=pd.DataFrame.from_records(
    poi_records,
    columns=['poi_name','longitude','latitude'],
)
assert len(pois)==int(extraction['deduplicated_pois'])
print('worker POIs transferred to notebook:',len(pois))
print('private CSV committed by worker:',POI_PRIVATE)


In [ ]:
classified=s07g.classify_poi_names(pois)
matches=s07g.assign_pois_to_anchors(eligible,classified,max_distance_m=100.0)
metrics=s07g.build_anchor_lexical_metrics(eligible,matches)
aligned=s07g.align_mobility_roles(metrics,work,profiles)

comparisons=s07g.build_stable_secondary_lexical_comparisons(aligned)
comparison_summary=s07g.summarize_stable_secondary_comparisons(comparisons)
stable_category_summary=s07g.summarize_stable_secondary_categories(aligned)
category_summary=s07g.summarize_category_thresholds(metrics)
axis_summary=s07g.summarize_profile_axis_context(aligned)
lexical_status=s07g.lexical_status_summary(classified)

print('spatially extracted deduplicated POIs:',len(pois))
print('exact anchor-POI pairs <=100m:',len(matches))
print('anchors with any POI <=100m:',int(metrics['any_poi_within_100m'].sum()),'/',len(metrics))
print('anchors with any lexical signal <=100m:',int(metrics['any_lexical_signal_within_100m'].sum()),'/',len(metrics))
print('anchors with work-compatible lexical signal <=100m:',int(metrics['work_compatible_lexical_within_100m'].sum()),'/',len(metrics))
print('stable-secondary BCL comparison rows:',len(comparisons))


In [ ]:
print('LEXICAL STATUS — EXTRACTED BCL POIs')
display(lexical_status)

print('LEXICAL CATEGORY × DISTANCE THRESHOLD')
display(category_summary)

print('STABLE SECONDARY — BCL LEXICAL SIGNAL VS SAME-USER PEERS')
display(comparison_summary)

print('STABLE SECONDARY — CATEGORY-SPECIFIC BCL LEXICAL SIGNAL')
display(stable_category_summary)

print('STAGE-07b AXES × BCL WORK-COMPATIBLE LEXICAL SIGNAL <=100m')
display(axis_summary)


In [ ]:
# Private precise artifacts
classified.to_pickle(C/'bcl_neighborhood_pois_classified_private.pkl')
matches.to_pickle(C/'anchor_poi_matches_private.pkl')
metrics.to_pickle(C/'anchor_bcl_lexical_metrics_private.pkl')
aligned.to_pickle(C/'mobility_aligned_bcl_context_private.pkl')
comparisons.to_pickle(C/'stable_secondary_bcl_comparisons_private.pkl')

# Aggregate/reproducibility outputs only
s07g.manifest_gate_frame(gate).to_csv(C/'source_manifest_gate.csv',index=False)
eligibility.to_csv(C/'bcl_eligibility_summary.csv',index=False)
pd.DataFrame([extraction]).to_csv(C/'spatial_extraction_summary.csv',index=False)
lexical_status.to_csv(C/'lexical_status_summary.csv',index=False)
category_summary.to_csv(C/'lexical_category_threshold_summary.csv',index=False)
comparison_summary.to_csv(C/'stable_secondary_bcl_summary.csv',index=False)
stable_category_summary.to_csv(C/'stable_secondary_bcl_category_summary.csv',index=False)
axis_summary.to_csv(C/'profile_axis_bcl_context_summary.csv',index=False)
print('saved:',C)


## Decision boundary

- Raw `PNAME` is preserved privately; lexical matches are deterministic signals, not trusted categories.
- `unknown` is a valid outcome. Multi-signal names remain ambiguous rather than being forced into one class.
- A positive lexical signal near an anchor is external historical context only.
- Stage 07g must not emit WORK/OFFICE/HOME labels or infer occupation/employment.
- The strongest check is the exact stable-secondary candidate versus same-user BCL-eligible recurring peers, with support counts and bootstrap uncertainty reported explicitly.
